# Creating LIB table
## Marissa Burton

### Load in packages

In [51]:
from glob import glob
import pandas as pd

In [52]:
CORPUS = pd.read_csv('data/CORPUS.csv')

/var/folders/k2/xkhh4x_x2999dprlhrz1h_r00000gn/T/ipykernel_94135/1873822874.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  CORPUS = pd.read_csv('data/CORPUS.csv')


### Load in file paths

In [53]:
source_file_list = sorted(glob('data/source_file_dir/*'))

### Create book data

In [54]:
book_data = []
for source_file_path in source_file_list:
    book_id = int(source_file_path.split('-')[-1].split('.')[0].replace('pg',''))
    raw_title = source_file_path.split('/')[-1].split('-')[0].replace('_', ' ')
    book_data.append((book_id, source_file_path, raw_title))

# Show example
book_data[0]

(1952,
 'data/source_file_dir/GILMAN_CHARLOTTE_PERKINS_THE_YELLOW_WALLPAPER-pg1952.txt',
 'GILMAN CHARLOTTE PERKINS THE YELLOW WALLPAPER')

### Create LIB table

In [55]:
LIB = pd.DataFrame(book_data, columns=['book_id','source_file_path', 'raw_title'])\
    .set_index('book_id').sort_index()

#### Fix author and title values for LIB table

In [56]:
try:
    LIB['author'] = LIB.raw_title.apply(lambda x: ', '.join(x.split()[:2]))
    LIB['title'] = LIB.raw_title.apply(lambda x: ' '.join(x.split()[2:]))
    LIB = LIB.drop('raw_title', axis=1)

# Skip if this cell has already been run
except AttributeError:
    pass

LIB.loc[43, 'author'] = 'STEVENSON, ROBERT LOUIS'
LIB.loc[1952, 'author'] = 'GILMAN, CHARLOTTE PERKINS'
LIB.loc[3268, 'author'] = 'RADCLIFFE, ANN WARD'
LIB.loc[6087, 'author'] = 'POLIDORI, JOHN WILLIAM'
LIB.loc[12122, 'author'] = 'JACOBS, W.W.'
LIB.loc[41445, 'author'] = 'SHELLEY, MARY WOLLSTONECRAFT'
LIB.loc[53685, 'author'] = 'MATURIN, CHARLES ROBERT'

LIB.loc[43, 'title'] = 'THE STRANGE CASE OF DR JEKYLL AND MR HYDE'
LIB.loc[1952, 'title'] = 'THE YELLOW WALLPAPER'
LIB.loc[3268, 'title'] = 'THE MYSTERIES OF UDOLPHO'
LIB.loc[6087, 'title'] = 'THE VAMPYRE'
LIB.loc[12122, 'title'] = "THE MONKEY'S PAW"
LIB.loc[41445, 'title'] = 'FRANKENSTEIN'
LIB.loc[53685, 'title'] = 'MELMOTH THE WANDERER VOL. 1'

#### Add book length

In [57]:
LIB['book_len'] = CORPUS.groupby('book_id').term_str.count()

#### Add number of chapters

In [58]:
LIB['n_chaps'] = CORPUS.reset_index()[['book_id','chap_num']]\
    .drop_duplicates()\
    .groupby('book_id').chap_num.count()

#### Add eras

In [59]:
LIB['era'] = CORPUS.groupby('book_id')['era'].first()

In [60]:
cols = LIB.columns.tolist()

cols.insert(1, cols.pop(cols.index('era')))

LIB = LIB[cols]

#### Get average book length

In [61]:
LIB['book_len'].mean()

np.float64(53639.333333333336)

### Export LIB table

In [62]:
LIB.to_csv('data/LIB.csv')